<a href="https://www.kaggle.com/code/sofiatanganho/2-0-modela-o-treino?scriptVersionId=313839321" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Importação de Bibliotecas

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.stats import kstest, norm
from sklearn.model_selection import train_test_split, KFold, ParameterGrid
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from sklearn.svm import SVR
import os
import warnings
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/natachazhang/dados-processados/boston_processed.csv")
df.info()


# Problemas Supervisionados

## Modelos de Regressão 

### Experimentação de valores para treino de 80% e teste de 20%

In [ ]:
X = df[[
    'ZN_stand',      
    'INDUS_norm',   
    'CHAS',          
    'AGE_norm',      
    'TAX_norm',     
    'B_stand',       
    'IQV_stand',    
    'IAH_stand'     
]]

y = df['MEDV']

# Divisão 80% treino / 20% teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Treino: {X_train.shape}")
print(f"Teste: {X_test.shape}")

In [ ]:
# Treinar Baseline
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)

# Avaliar
rmse_bl = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
mae_bl = mean_absolute_error(y_test, y_pred_baseline)
r2_bl = r2_score(y_test, y_pred_baseline)
print('Regressão Linear')
print(f"RMSE: {rmse_bl:.4f}")
print(f"MAE:  {mae_bl:.4f}")
print(f"R²:   {r2_bl:.4f}")

In [ ]:
# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf_train = rf.predict(X_train)
y_pred_rf_test = rf.predict(X_test)

rmse_rf_train = np.sqrt(mean_squared_error(y_train, y_pred_rf_train))
rmse_rf_test = np.sqrt(mean_squared_error(y_test, y_pred_rf_test))
mae_rf_train = mean_absolute_error(y_train, y_pred_rf_train)
mae_rf_test = mean_absolute_error(y_test, y_pred_rf_test)
r2_rf_train = r2_score(y_train, y_pred_rf_train)
r2_rf_test = r2_score(y_test, y_pred_rf_test)

print("Random Forest")
print(f"RMSE Treino: {rmse_rf_train:.4f} | RMSE Teste: {rmse_rf_test:.4f}")
print(f"MAE  Treino: {mae_rf_train:.4f} | MAE  Teste: {mae_rf_test:.4f}")
print(f"R²   Treino: {r2_rf_train:.4f} | R²   Teste: {r2_rf_test:.4f}")

In [ ]:
# XGBoost
xgb = XGBRegressor(n_estimators=100, random_state=42)
xgb.fit(X_train, y_train)
y_pred_xgb_train = xgb.predict(X_train)
y_pred_xgb_test = xgb.predict(X_test)

rmse_xgb_train = np.sqrt(mean_squared_error(y_train, y_pred_xgb_train))
rmse_xgb_test = np.sqrt(mean_squared_error(y_test, y_pred_xgb_test))
mae_xgb_train = mean_absolute_error(y_train, y_pred_xgb_train)
mae_xgb_test = mean_absolute_error(y_test, y_pred_xgb_test)
r2_xgb_train = r2_score(y_train, y_pred_xgb_train)
r2_xgb_test = r2_score(y_test, y_pred_xgb_test)

print("XGBOOST")
print(f"RMSE Treino: {rmse_xgb_train:.4f} | RMSE Teste: {rmse_xgb_test:.4f}")
print(f"MAE  Treino: {mae_xgb_train:.4f} | MAE  Teste: {mae_xgb_test:.4f}")
print(f"R²   Treino: {r2_xgb_train:.4f} | R²   Teste: {r2_xgb_test:.4f}")

In [ ]:
# SVR precisa de dados normalizados
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Treinar SVR
svr = SVR(kernel='rbf')
svr.fit(X_train_scaled, y_train)

# Métricas Treino
y_pred_svr_train = svr.predict(X_train_scaled)
rmse_svr_train = np.sqrt(mean_squared_error(y_train, y_pred_svr_train))
mae_svr_train = mean_absolute_error(y_train, y_pred_svr_train)
r2_svr_train = r2_score(y_train, y_pred_svr_train)

# Métricas Teste
y_pred_svr_test = svr.predict(X_test_scaled)
rmse_svr_test = np.sqrt(mean_squared_error(y_test, y_pred_svr_test))
mae_svr_test = mean_absolute_error(y_test, y_pred_svr_test)
r2_svr_test = r2_score(y_test, y_pred_svr_test)

print("SVR")
print(f"RMSE Treino: {rmse_svr_train:.4f} | RMSE Teste: {rmse_svr_test:.4f}")
print(f"MAE  Treino: {mae_svr_train:.4f} | MAE  Teste: {mae_svr_test:.4f}")
print(f"R²   Treino: {r2_svr_train:.4f} | R²   Teste: {r2_svr_test:.4f}")

In [ ]:
resultados = pd.DataFrame({
    'Modelo': ['Random Forest', 'XGBoost', 'SVR'],
    'RMSE Treino': [ rmse_rf_train, rmse_xgb_train, rmse_svr_train],
    'RMSE Teste':  [ rmse_rf_test, rmse_xgb_test, rmse_svr_test],
    'MAE Treino':  [ mae_rf_train, mae_xgb_train, mae_svr_train],
    'MAE Teste':   [ mae_rf_test, mae_xgb_test, mae_svr_test],
    'R² Treino':   [ r2_rf_train, r2_xgb_train, r2_svr_train],
    'R² Teste':    [ r2_rf_test, r2_xgb_test, r2_svr_test]
})

print(resultados.to_string(index=False))

### Experimentação de valores para treino de 70% e teste de 30%

In [ ]:
X = df[[
    'ZN_stand',      
    'INDUS_norm',    
    'CHAS',          
    'AGE_norm',      
    'TAX_norm',      
    'B_stand',       
    'IQV_stand',   
    'IAH_stand'      
]]

y = df['MEDV']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Treino: {X_train.shape}")
print(f"Teste: {X_test.shape}")

In [ ]:
# Treinar Baseline
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)


rmse_bl = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
mae_bl = mean_absolute_error(y_test, y_pred_baseline)
r2_bl = r2_score(y_test, y_pred_baseline)

print("Regressão Linear")
print(f"RMSE: {rmse_bl:.4f}")
print(f"MAE:  {mae_bl:.4f}")
print(f"R²:   {r2_bl:.4f}")

In [ ]:
# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf_train = rf.predict(X_train)
y_pred_rf_test = rf.predict(X_test)

rmse_rf_train = np.sqrt(mean_squared_error(y_train, y_pred_rf_train))
rmse_rf_test = np.sqrt(mean_squared_error(y_test, y_pred_rf_test))
mae_rf_train = mean_absolute_error(y_train, y_pred_rf_train)
mae_rf_test = mean_absolute_error(y_test, y_pred_rf_test)
r2_rf_train = r2_score(y_train, y_pred_rf_train)
r2_rf_test = r2_score(y_test, y_pred_rf_test)

print("RANDOM FOREST")
print(f"RMSE Treino: {rmse_rf_train:.4f} | RMSE Teste: {rmse_rf_test:.4f}")
print(f"MAE  Treino: {mae_rf_train:.4f} | MAE  Teste: {mae_rf_test:.4f}")
print(f"R²   Treino: {r2_rf_train:.4f} | R²   Teste: {r2_rf_test:.4f}")

In [ ]:
# XGBoost
xgb = XGBRegressor(n_estimators=100, random_state=42)
xgb.fit(X_train, y_train)
y_pred_xgb_train = xgb.predict(X_train)
y_pred_xgb_test = xgb.predict(X_test)

rmse_xgb_train = np.sqrt(mean_squared_error(y_train, y_pred_xgb_train))
rmse_xgb_test = np.sqrt(mean_squared_error(y_test, y_pred_xgb_test))
mae_xgb_train = mean_absolute_error(y_train, y_pred_xgb_train)
mae_xgb_test = mean_absolute_error(y_test, y_pred_xgb_test)
r2_xgb_train = r2_score(y_train, y_pred_xgb_train)
r2_xgb_test = r2_score(y_test, y_pred_xgb_test)

print("XGBOOST")
print(f"RMSE Treino: {rmse_xgb_train:.4f} | RMSE Teste: {rmse_xgb_test:.4f}")
print(f"MAE  Treino: {mae_xgb_train:.4f} | MAE  Teste: {mae_xgb_test:.4f}")
print(f"R²   Treino: {r2_xgb_train:.4f} | R²   Teste: {r2_xgb_test:.4f}")

In [ ]:
# SVR precisa de dados normalizados
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Treinar SVR
svr = SVR(kernel='rbf')
svr.fit(X_train_scaled, y_train)

# Métricas Treino
y_pred_svr_train = svr.predict(X_train_scaled)
rmse_svr_train = np.sqrt(mean_squared_error(y_train, y_pred_svr_train))
mae_svr_train = mean_absolute_error(y_train, y_pred_svr_train)
r2_svr_train = r2_score(y_train, y_pred_svr_train)

# Métricas Teste
y_pred_svr_test = svr.predict(X_test_scaled)
rmse_svr_test = np.sqrt(mean_squared_error(y_test, y_pred_svr_test))
mae_svr_test = mean_absolute_error(y_test, y_pred_svr_test)
r2_svr_test = r2_score(y_test, y_pred_svr_test)

print("SVR")
print(f"RMSE Treino: {rmse_svr_train:.4f} | RMSE Teste: {rmse_svr_test:.4f}")
print(f"MAE  Treino: {mae_svr_train:.4f} | MAE  Teste: {mae_svr_test:.4f}")
print(f"R²   Treino: {r2_svr_train:.4f} | R²   Teste: {r2_svr_test:.4f}")

In [ ]:
# Tabela Comparativa
resultados = pd.DataFrame({
    'Modelo': ['Random Forest', 'XGBoost', 'SVR'],
    'RMSE Treino': [ rmse_rf_train, rmse_xgb_train, rmse_svr_train],
    'RMSE Teste':  [ rmse_rf_test, rmse_xgb_test, rmse_svr_test],
    'MAE Treino':  [ mae_rf_train, mae_xgb_train, mae_svr_train],
    'MAE Teste':   [ mae_rf_test, mae_xgb_test, mae_svr_test],
    'R² Treino':   [ r2_rf_train, r2_xgb_train, r2_svr_train],
    'R² Teste':    [ r2_rf_test, r2_xgb_test, r2_svr_test]
})

print(resultados.to_string(index=False))

# Problemas Não Supervisionados

## Modelos de Agrupamentos (_Clustering_)

In [ ]:
FIG_DIR = "reports/figures"
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
feature_priority = {
    "ZN": ["ZN_stand", "ZN_norm", "ZN"],
    "INDUS": ["INDUS_stand", "INDUS_norm", "INDUS"],
    "CHAS": ["CHAS"],
    "AGE": ["AGE_stand", "AGE_norm", "AGE"],
    "TAX": ["TAX_stand", "TAX_norm", "TAX"],
    "B": ["B_stand", "B_norm", "B"],
    "IQV": ["IQV_stand", "IQV_norm", "IQV"],
    "IAH": ["IAH_stand", "IAH_norm", "IAH"],
}


selected_features = []
feature_origin_map = {}

for base_name, candidates in feature_priority.items():
    chosen = next((col for col in candidates if col in df.columns), None)
    if chosen is not None:
        selected_features.append(chosen)
        feature_origin_map[chosen] = base_name

if not selected_features:
    raise ValueError("Nenhuma das colunas esperadas para clustering foi encontrada no dataframe.")

X = df[selected_features].copy()

print("Variáveis usadas no clustering:")
print(selected_features)
print("\nShape do dataset de clustering:", X.shape)

# Divisão em treino e teste para avaliar a estabilidade

In [ ]:
X_train, X_test = train_test_split(X, test_size=0.30, random_state=42)

print("\nTrain:", X_train.shape)
print("Test :", X_test.shape)


In [ ]:
import numpy as np
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)


def get_valid_labels(labels):
    labels = np.asarray(labels)
    mask = labels != -1  # remove ruído do DBSCAN
    valid_labels = labels[mask]
    return mask, valid_labels


def safe_silhouette(X_part, labels):
    X_part = np.asarray(X_part)
    labels = np.asarray(labels)

    mask, valid_labels = get_valid_labels(labels)
    n_clusters = len(set(valid_labels))

    if valid_labels.size == 0 or n_clusters < 2:
        return np.nan

    return silhouette_score(X_part[mask], valid_labels)


def safe_calinski(X_part, labels):
    X_part = np.asarray(X_part)
    labels = np.asarray(labels)

    mask, valid_labels = get_valid_labels(labels)
    n_clusters = len(set(valid_labels))

    if valid_labels.size == 0 or n_clusters < 2:
        return np.nan

    return calinski_harabasz_score(X_part[mask], valid_labels)


def safe_davies(X_part, labels):
    X_part = np.asarray(X_part)
    labels = np.asarray(labels)

    mask, valid_labels = get_valid_labels(labels)
    n_clusters = len(set(valid_labels))

    if valid_labels.size == 0 or n_clusters < 2:
        return np.nan

    return davies_bouldin_score(X_part[mask], valid_labels)


def count_clusters(labels):
    labels = np.asarray(labels)
    unique = set(labels)
    if -1 in unique:
        unique.remove(-1)
    return len(unique)



def evaluate_on_split(model, X_part):
    labels = model.fit_predict(X_part)

    silhouette = safe_silhouette(X_part, labels)
    calinski = safe_calinski(X_part, labels)
    davies = safe_davies(X_part, labels)

    n_clusters = count_clusters(labels)
    noise = int(np.sum(np.asarray(labels) == -1))
    inertia = getattr(model, "inertia_", np.nan)

    return {
        "labels": labels,
        "silhouette": silhouette,
        "calinski": calinski,
        "davies": davies,
        "n_clusters": n_clusters,
        "noise_points": noise,
        "inertia": inertia
    }


def evaluate_train_test(model_train, model_test, model_name):
    train_res = evaluate_on_split(model_train, X_train)
    test_res  = evaluate_on_split(model_test, X_test)

    return {
        "Modelo": model_name,

        "Silhouette_Treino": train_res["silhouette"],
        "Silhouette_Teste": test_res["silhouette"],

        "Calinski_Treino": train_res["calinski"],
        "Calinski_Teste": test_res["calinski"],

        "Davies_Treino": train_res["davies"],
        "Davies_Teste": test_res["davies"],

        "Clusters_Treino": train_res["n_clusters"],
        "Clusters_Teste": test_res["n_clusters"],

        "Ruido_Treino": train_res["noise_points"],
        "Ruido_Teste": test_res["noise_points"],

        "Inertia_Treino": train_res["inertia"],
        "Inertia_Teste": test_res["inertia"],

        "Labels_Treino": train_res["labels"],
        "Labels_Teste": test_res["labels"]
    }

# Baseline - KMeans

In [ ]:
baseline_model = KMeans(random_state=42, n_init="auto")

baseline_res = evaluate_train_test(
    baseline_model,
    KMeans(random_state=42, n_init="auto"),
    "Baseline - KMeans default"
)

print("Kmeans")

# Silhouette
print(f"Silhouette Treino: {baseline_res['Silhouette_Treino']:.4f}")
print(f"Silhouette Teste : {baseline_res['Silhouette_Teste']:.4f}")

# Calinski-Harabasz
print(f"Calinski-Harabasz Treino: {baseline_res['Calinski_Treino']:.4f}")
print(f"Calinski-Harabasz Teste : {baseline_res['Calinski_Teste']:.4f}")

# Davies-Bouldin
print(f"Davies-Bouldin Treino: {baseline_res['Davies_Treino']:.4f}")
print(f"Davies-Bouldin Teste : {baseline_res['Davies_Teste']:.4f}")

# Clusters
print(f"Clusters Treino  : {baseline_res['Clusters_Treino']}")
print(f"Clusters Teste   : {baseline_res['Clusters_Teste']}")

In [ ]:
k_values = range(2, 11)
elbow_rows = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = km.fit_predict(X_train)

    elbow_rows.append({
        "k": k,
        "inertia": km.inertia_,
        "silhouette": safe_silhouette(X_train, labels)
    })

elbow_df = pd.DataFrame(elbow_rows)

best_k = int(elbow_df.loc[elbow_df["silhouette"].idxmax(), "k"])
best_k_sil = elbow_df["silhouette"].max()

print("ELBOW / SILHOUETTE (KMEANS)")
print(elbow_df.to_string(index=False))
print(f"\nMelhor k pelo Silhouette Score: {best_k} (score={best_k_sil:.4f})")

plt.figure(figsize=(8, 5))
plt.plot(elbow_df["k"], elbow_df["inertia"], marker="o")
plt.title("Método do Cotovelo - KMeans")
plt.xlabel("Número de clusters (k)")
plt.ylabel("Inertia (WCSS)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/elbow_method_kmeans.png", dpi=300)
plt.show()

In [ ]:
# KMeans otimizado
results_df = pd.DataFrame([{
    "Métrica": "Silhouette Score",
    "Treino": kmeans_tuned_res["Silhouette_Treino"],
    "Teste": kmeans_tuned_res["Silhouette_Teste"]
}, {
    "Métrica": "Calinski-Harabasz",
    "Treino": kmeans_tuned_res["Calinski_Treino"],
    "Teste": kmeans_tuned_res["Calinski_Teste"]
}, {
    "Métrica": "Davies-Bouldin",
    "Treino": kmeans_tuned_res["Davies_Treino"],
    "Teste": kmeans_tuned_res["Davies_Teste"]
}])

print(results_df.to_string(index=False))

Com o valor ótimo de k definido, treinou-se o modelo KMeans final com 4 clusters. As métricas obtidas revelam um desempenho consistente entre treino e teste, o que indica boa capacidade de generalização da segmentação. O Silhouette Score mantém-se elevado em ambos os conjuntos (0.5545 no treino e 0.5364 no teste), o índice de Calinski-Harabasz confirma clusters bem separados, e o Davies-Bouldin, onde valores mais baixos são preferíveis, apresenta resultados igualmente satisfatórios. A coerência entre as métricas de treino e teste sugere que o modelo não está sobreajustado.

In [ ]:
# Agglomerative
agg_results = []
for params in ParameterGrid({
    "n_clusters": list(range(2, 11)),
    "linkage": ["ward", "complete", "average"]
}):
    model_name = f"Agglomerative {params}"
    res = evaluate_train_test(
        AgglomerativeClustering(**params),
        AgglomerativeClustering(**params),
        model_name
    )
    agg_results.append((params, res))

best_agg_params, best_agg_res = max(
    agg_results,
    key=lambda x: -999 if np.isnan(x[1]["Silhouette_Teste"]) else x[1]["Silhouette_Teste"]
)

print("AGGLOMERATIVE")

print(f"Params: {best_agg_params}")

print(f"Silhouette Treino: {best_agg_res['Silhouette_Treino']:.4f}")
print(f"Silhouette Teste : {best_agg_res['Silhouette_Teste']:.4f}")

print(f"Calinski-Harabasz Treino: {best_agg_res['Calinski_Treino']:.4f}")
print(f"Calinski-Harabasz Teste : {best_agg_res['Calinski_Teste']:.4f}")

print(f"Davies-Bouldin Treino: {best_agg_res['Davies_Treino']:.4f}")
print(f"Davies-Bouldin Teste : {best_agg_res['Davies_Teste']:.4f}")

print(f"Clusters Treino: {best_agg_res['Clusters_Treino']}")
print(f"Clusters Teste : {best_agg_res['Clusters_Teste']}")

print(f"Ruído Treino: {best_agg_res['Ruido_Treino']}")
print(f"Ruído Teste : {best_agg_res['Ruido_Teste']}")

In [ ]:
# DBSCAN 
dbscan_results = []
for params in ParameterGrid({
    "eps": [0.3, 0.5, 0.7, 0.9, 1.1, 1.3],
    "min_samples": [3, 5, 8, 10]
}):
    model_name = f"DBSCAN {params}"
    res = evaluate_train_test(
        DBSCAN(**params),
        DBSCAN(**params),
        model_name
    )
    dbscan_results.append((params, res))

best_dbscan_params, best_dbscan_res = max(
    dbscan_results,
    key=lambda x: -999 if np.isnan(x[1]["Silhouette_Teste"]) else x[1]["Silhouette_Teste"]
)

print("DBSCAN MODEL")

print(f"Params: {best_dbscan_params}")

print(f"Silhouette Treino: {best_dbscan_res['Silhouette_Treino']:.4f}")
print(f"Silhouette Teste : {best_dbscan_res['Silhouette_Teste']:.4f}")

print(f"Calinski-Harabasz Treino: {best_dbscan_res['Calinski_Treino']:.4f}")
print(f"Calinski-Harabasz Teste : {best_dbscan_res['Calinski_Teste']:.4f}")

print(f"Davies-Bouldin Treino: {best_dbscan_res['Davies_Treino']:.4f}")
print(f"Davies-Bouldin Teste : {best_dbscan_res['Davies_Teste']:.4f}")

print(f"Clusters Treino: {best_dbscan_res['Clusters_Treino']}")
print(f"Clusters Teste : {best_dbscan_res['Clusters_Teste']}")

print(f"Ruído Treino: {best_dbscan_res['Ruido_Treino']}")
print(f"Ruído Teste : {best_dbscan_res['Ruido_Teste']}")

In [ ]:
#  TABELA COMPARATIVA

comparison_df = pd.DataFrame([
    {k: v for k, v in baseline_res.items() if not k.startswith("Labels")},
    {k: v for k, v in kmeans_tuned_res.items() if not k.startswith("Labels")},
    {k: v for k, v in best_agg_res.items() if not k.startswith("Labels")},
    {k: v for k, v in best_dbscan_res.items() if not k.startswith("Labels")},
])

comparison_df["Gap_Treino_Teste"] = (
    comparison_df["Silhouette_Treino"] - comparison_df["Silhouette_Teste"]
).abs()

comparison_df = comparison_df.sort_values(
    by=["Silhouette_Teste", "Gap_Treino_Teste"],
    ascending=[False, True]
).reset_index(drop=True)

print("TABELA COMPARATIVA ")
print(comparison_df.to_string(index=False))

best_model_name = comparison_df.loc[0, "Modelo"]
print(f"\nMelhor modelo inicial: {best_model_name}")

# Recriar o melhor modelo para o dataset completo

In [ ]:
if "KMeans otimizado" in best_model_name:
    final_model = KMeans(n_clusters=best_k, random_state=42, n_init="auto")
elif "Baseline - KMeans default" in best_model_name:
    final_model = KMeans(random_state=42, n_init="auto")
elif "Agglomerative" in best_model_name:
    final_model = AgglomerativeClustering(**best_agg_params)
else:
    final_model = DBSCAN(**best_dbscan_params)

final_labels = final_model.fit_predict(X)
final_silhouette = safe_silhouette(X, final_labels)
final_n_clusters = count_clusters(final_labels)

print(" Modelo final")
print("Modelo:", final_model)
print(f"Silhouette final: {final_silhouette:.4f}")
print(f"Número de clusters: {final_n_clusters}")

In [ ]:
# VALIDAÇÃO CRUZADA / ESTABILIDADE (K-FOLD)

def clone_best_model():
    if "KMeans otimizado" in best_model_name:
        return KMeans(n_clusters=best_k, random_state=42, n_init="auto")
    if "Baseline - KMeans default" in best_model_name:
        return KMeans(random_state=42, n_init="auto")
    if "Agglomerative" in best_model_name:
        return AgglomerativeClustering(**best_agg_params)
    return DBSCAN(**best_dbscan_params)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (_, test_idx) in enumerate(kf.split(X), start=1):
    X_fold = X.iloc[test_idx]
    model = clone_best_model()
    labels_fold = model.fit_predict(X_fold)
    score_fold = safe_silhouette(X_fold, labels_fold)
    cv_scores.append(score_fold)
    print(f"Fold {fold}: Silhouette = {score_fold:.4f}")

cv_scores = np.array(cv_scores, dtype=float)
cv_mean = np.nanmean(cv_scores)
cv_std = np.nanstd(cv_scores)

print(f"\nCV Silhouette Média: {cv_mean:.4f}")
print(f"CV Silhouette Desv.Padrão: {cv_std:.4f}")

In [ ]:
# SILHOUETTE PLOT DO MODELO FINAL

def silhouette_plot(X_data, labels, save_path):
    X_data = np.asarray(X_data)
    labels = np.asarray(labels)

    mask, valid_labels = get_valid_labels(labels)
    X_valid = X_data[mask]

    if len(set(valid_labels)) < 2:
        print("Não foi possível gerar Silhouette Plot: menos de 2 clusters válidos.")
        return

    sample_silhouette_values = silhouette_samples(X_valid, valid_labels)
    unique_clusters = sorted(set(valid_labels))

    fig, ax = plt.subplots(figsize=(9, 6))
    y_lower = 10

    for cluster_id in unique_clusters:
        cluster_vals = sample_silhouette_values[valid_labels == cluster_id]
        cluster_vals.sort()
        size_cluster = cluster_vals.shape[0]
        y_upper = y_lower + size_cluster

        ax.fill_betweenx(
            np.arange(y_lower, y_upper),
            0, cluster_vals,
            alpha=0.7
        )
        ax.text(-0.05, y_lower + 0.5 * size_cluster, str(cluster_id))
        y_lower = y_upper + 10

    avg_score = silhouette_score(X_valid, valid_labels)
    ax.axvline(x=avg_score, linestyle="--")
    ax.set_title("Silhouette Plot - Modelo Final")
    ax.set_xlabel("Coeficiente de Silhueta")
    ax.set_ylabel("Cluster")
    ax.set_yticks([])
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.show()

silhouette_plot(X, final_labels, f"{FIG_DIR}/silhouette_plot_modelo_final.png")


In [ ]:
# VISUALIZAÇÃO 2D DOS SEGMENTOS (PCA)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=final_labels, alpha=0.8)
plt.title("Visualização dos Segmentos (PCA 2D)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/clusters_pca.png", dpi=300)
plt.show()

In [ ]:
# PERFIL DOS SEGMENTOS

profile_df = pd.DataFrame()
for transformed_col in selected_features:
    base_name = feature_origin_map[transformed_col]
    if base_name in df.columns:
        profile_df[base_name] = df[base_name]
    else:
        profile_df[transformed_col] = df[transformed_col]

profile_df["Cluster"] = final_labels

# remover ruído do DBSCAN na perfilagem, se existir
profile_df_valid = profile_df[profile_df["Cluster"] != -1].copy()

cluster_profile = profile_df_valid.groupby("Cluster").mean(numeric_only=True)
print("PERFIL MÉDIO DOS CLUSTERS")
print(cluster_profile)

cluster_profile.T.plot(kind="bar", figsize=(12, 6))
plt.title("Perfil Médio dos Segmentos")
plt.xlabel("Variáveis")
plt.ylabel("Valor médio")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/perfil_segmentos.png", dpi=300)
plt.show()

In [ ]:
# DATAFRAME FINAL COM CLUSTERS

df_clusterizado = df.copy()
df_clusterizado["Cluster"] = final_labels

print("\nContagem por cluster:")
print(df_clusterizado["Cluster"].value_counts().sort_index())

In [ ]:
# Resumo
summary = {
    "baseline_modelo": baseline_res["Modelo"],
    "baseline_silhouette_teste": baseline_res["Silhouette_Teste"],
    "melhor_modelo": best_model_name,
    "silhouette_final": final_silhouette,
    "n_clusters_final": final_n_clusters,
    "cv_mean_silhouette": cv_mean,
    "cv_std_silhouette": cv_std,
    "best_k_kmeans": best_k,
    "best_agg_params": best_agg_params,
    "best_dbscan_params": best_dbscan_params
}

print("RESUMO ")
for k, v in summary.items():
    print(f"{k}: {v}")